# Transformer via DSL — `arch` declaration

One `compile()` call produces an `ArchDef`. One `arch.interpreter('Transformer')`
returns an `ArchInterpreter` with both `.run_algebra()` and `.run_coalgebra()`.

What the DSL declares:

- **Semiring** — the contract op that all legs share.
- **Legs** — typed sorts, einsum equations, per-leg op overrides.
- **Paths** — `read` and `mlp` as alias compositions.
- **Fan-out** — `kv` runs `k_proj` and `v_proj` in parallel.
- **`arch Transformer:`** — algebra cases (tree fold) and coalgebra cases
  (streaming unfold), each with bound cell functions.

What remains as Python:

- Utility functions (softmax, layer_norm, gelu, causal_mask).
- Per-leg op implementations — arbitrary numpy, referenced by dotted names.
- Cell functions — per-case computation logic, referenced by dotted names.
- Weight initialization and bundle assembly.

In [1]:
import sys, os
import numpy as np

ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)
sys.modules.pop("engine", None)

from engine import compile, CoalgResult

np.set_printoptions(precision=4, suppress=True)

## 1. Hyperparameters and utilities

In [2]:
V = 128
D = 48
N = 4
H = D // N
F = D * 3
L = 2
S = 8
EPS = 1e-5
SCALE = H ** -0.5

assert D % N == 0

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def softmax_masked(x: np.ndarray, mask: np.ndarray | None = None, axis: int = -1) -> np.ndarray:
    if mask is not None:
        x = x + mask
    return softmax(x, axis=axis)

def layer_norm(x: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = EPS) -> np.ndarray:
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return ((x - mu) / np.sqrt(var + eps)) * gamma + beta

def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

def causal_mask(t: int) -> np.ndarray:
    return np.triu(np.full((t, t), -1e9), 1)

print({'vocab': V, 'd_model': D, 'heads': N, 'd_head': H, 'd_ff': F, 'layers': L, 'seq_len': S})

{'vocab': 128, 'd_model': 48, 'heads': 4, 'd_head': 12, 'd_ff': 144, 'layers': 2, 'seq_len': 8}


## 2. Ops, cell functions, and DSL architecture

Everything the DSL references lives in a single namespace:
- **Leg ops** — `(compiled_eq, x, bundle, temp) -> result`
- **Cell functions** — algebra cells (`input_cell`, `attn_cell`, `ffn_cell`, `norm_cell`)
  and one coalgebra cell (`stream_cell`). These receive compiled paths through
  `params['paths']`, injected automatically by `arch.interpreter()`.

The `arch Transformer:` block at the end declares both the algebra and coalgebra
sides in a single block.

In [3]:
# ---------------------------------------------------------------------------
# Per-leg op implementations (plain numpy)
# ---------------------------------------------------------------------------

def q_proj_op(eq, x, bundle, temp=0.0):
    return np.einsum(eq, x, bundle['Wq']) + bundle['bq']

def k_proj_op(eq, x, bundle, temp=0.0):
    return np.einsum(eq, x, bundle['Wk']) + bundle['bk']

def v_proj_op(eq, x, bundle, temp=0.0):
    return np.einsum(eq, x, bundle['Wv']) + bundle['bv']

def out_proj_op(eq, x, bundle, temp=0.0):
    return np.einsum(eq, x, bundle['Wo']) + bundle['bo']

def score_op(eq, q, bundle, temp=0.0):
    return np.einsum(eq, q, bundle['K']) * bundle['scale']

def normalize_op(eq, scores, bundle, temp=0.0):
    return softmax_masked(scores, bundle.get('mask'))

def mix_op(eq, probs, bundle, temp=0.0):
    return np.einsum(eq, probs, bundle['V'])

def up_op(eq, x, bundle, temp=0.0):
    return np.einsum(eq, x, bundle['W1']) + bundle['b1']

def act_op(eq, x, bundle, temp=0.0):
    return gelu(x)

def down_op(eq, x, bundle, temp=0.0):
    return np.einsum(eq, x, bundle['W2']) + bundle['b2']

def identity_op(eq, x, y=None, temp=0.0):
    return x


# ---------------------------------------------------------------------------
# Per-case algebra cells — receive paths via params['paths']
# ---------------------------------------------------------------------------

def _build_attn_bundle(paths, x_norm, w, mask):
    """Assemble the attention bundle, using the 'kv' fan for K/V projections."""
    bundle = {
        'Wq': w['Wq'], 'bq': w['bq'],
        'Wk': w['Wk'], 'bk': w['bk'],
        'Wv': w['Wv'], 'bv': w['bv'],
        'Wo': w['Wo'], 'bo': w['bo'],
        'scale': SCALE, 'mask': mask,
    }
    kv = paths['kv'](x_norm, bundle, 0.0)
    bundle['K'] = kv['k_proj']
    bundle['V'] = kv['v_proj']
    return bundle

def input_cell(payload, child_results, params, temp):
    return payload[0]

def attn_cell(payload, child_results, params, temp):
    w = payload[0]
    x = child_results[0]
    paths = params['paths']
    x_norm = layer_norm(x, w['ln1_g'], w['ln1_b'])
    bundle = _build_attn_bundle(paths, x_norm, w, params['mask'])
    return x + paths['read'](x_norm, bundle, 0.0)

def ffn_cell(payload, child_results, params, temp):
    w = payload[0]
    x = child_results[0]
    paths = params['paths']
    x_norm = layer_norm(x, w['ln2_g'], w['ln2_b'])
    return x + paths['mlp'](x_norm, w, 0.0)

def norm_cell(payload, child_results, params, temp):
    gamma, beta = payload
    return layer_norm(child_results[0], gamma, beta)


# ---------------------------------------------------------------------------
# Coalgebra cell — one function, dispatches on event mode
# ---------------------------------------------------------------------------

def stream_cell(state, event, params, temp):
    paths = params['paths']
    mode, token = event
    pos = state['pos']
    caches = state['caches']

    x = params['tok_embed'][token] + params['pos_enc'][pos]
    x = x[None, :]

    next_caches = []
    for layer_idx, w in enumerate(params['layer_weights']):
        x_norm = layer_norm(x, w['ln1_g'], w['ln1_b'])
        bundle = {
            'Wq': w['Wq'], 'bq': w['bq'],
            'Wk': w['Wk'], 'bk': w['bk'],
            'Wv': w['Wv'], 'bv': w['bv'],
            'Wo': w['Wo'], 'bo': w['bo'],
            'scale': SCALE, 'mask': None,
        }
        kv = paths['kv'](x_norm, bundle, 0.0)
        K_all = np.concatenate([caches[layer_idx]['K'], kv['k_proj']], axis=0)
        V_all = np.concatenate([caches[layer_idx]['V'], kv['v_proj']], axis=0)
        bundle['K'] = K_all
        bundle['V'] = V_all
        x = x + paths['read'](x_norm, bundle, 0.0)
        x = x + paths['mlp'](layer_norm(x, w['ln2_g'], w['ln2_b']), w, 0.0)
        next_caches.append({'K': K_all, 'V': V_all})

    logits = (layer_norm(x, params['final_ln_g'], params['final_ln_b']) @ params['unembed'])[0]
    next_state = {'pos': pos + 1, 'caches': next_caches}

    if mode == 'cache':
        return CoalgResult('cache_only', [pos], [next_state])
    return CoalgResult('emit', [pos], [next_state], output=logits)


# ---------------------------------------------------------------------------
# DSL architecture — single arch block with algebra + coalgebra
# ---------------------------------------------------------------------------

DSL_SOURCE = """
semiring attn:
    contract = ops.identity

sort model, q, scores, probs, mixed, kv, ff

leg q_proj    : model  -> q       via "sd,ndh->snh"   op ops.q_proj
leg score     : q      -> scores  via "snh,tnh->nst"  op ops.score
leg normalize : scores -> probs   via "nst->nst"       op ops.normalize
leg mix       : probs  -> mixed   via "nst,tnh->snh"  op ops.mix
leg out_proj  : mixed  -> model   via "snh,nhd->sd"   op ops.out_proj
leg k_proj    : model  -> kv      via "sd,ndh->snh"   op ops.k_proj
leg v_proj    : model  -> kv      via "sd,ndh->snh"   op ops.v_proj

leg up   : model -> ff     via "sd,df->sf"   op ops.up
leg act  : ff    -> ff     via "sf->sf"      op ops.act
leg down : ff    -> model  via "sf,fd->sd"   op ops.down

path read = q_proj score normalize mix out_proj
path mlp  = up act down

fan kv = k_proj & v_proj

arch Transformer:
    algebra:
        case input:         recursive=0  data=1  cell=ops.input_cell
        case attn_residual: recursive=1  data=1  cell=ops.attn_cell
        case ffn_residual:  recursive=1  data=1  cell=ops.ffn_cell
        case final_norm:    recursive=1  data=2  cell=ops.norm_cell
    coalgebra:
        cell = ops.stream_cell
        case cache_only: recursive=1  data=1  output=0
        case emit:       recursive=1  data=1  output=1
"""

arch = compile(DSL_SOURCE, {
    'ops': type('ns', (), {
        'identity':     staticmethod(identity_op),
        'q_proj':       staticmethod(q_proj_op),
        'k_proj':       staticmethod(k_proj_op),
        'v_proj':       staticmethod(v_proj_op),
        'out_proj':     staticmethod(out_proj_op),
        'score':        staticmethod(score_op),
        'normalize':    staticmethod(normalize_op),
        'mix':          staticmethod(mix_op),
        'up':           staticmethod(up_op),
        'act':          staticmethod(act_op),
        'down':         staticmethod(down_op),
        'input_cell':   staticmethod(input_cell),
        'attn_cell':    staticmethod(attn_cell),
        'ffn_cell':     staticmethod(ffn_cell),
        'norm_cell':    staticmethod(norm_cell),
        'stream_cell':  staticmethod(stream_cell),
    })(),
})

print("Compiled architecture:")
print(f"  paths: {sorted(arch.paths.keys())}")
print(f"  arch:  Transformer (algebra + coalgebra)")

Compiled architecture:
  paths: ['act', 'down', 'k_proj', 'kv', 'mix', 'mlp', 'normalize', 'out_proj', 'q_proj', 'read', 'score', 'up', 'v_proj']
  arch:  Transformer (algebra + coalgebra)


## 3. Path inspection

`ArchDef.explain()` shows the canonical leg sequence for any named path.
Alias paths (`read`, `mlp`) are expanded by the `AliasNormalizer` that
the DSL compiler wires in automatically.

In [4]:
print(arch.explain("read"))
print()
print(arch.explain("mlp"))

Path: read
Normal form: q_proj score normalize mix out_proj
  1. q_proj  [sd,ndh->snh]
  2. score  [snh,tnh->nst]
  3. normalize  [nst->nst]
  4. mix  [nst,tnh->snh]
  5. out_proj  [snh,nhd->sd]

Path: mlp
Normal form: up act down
  1. up  [sd,df->sf]
  2. act  [sf->sf]
  3. down  [sf,fd->sd]


## 4. Parameters

In [5]:
rng = np.random.default_rng(7)

def normal(shape, std=0.02):
    return rng.normal(0.0, std, shape)

def init_block(d: int, n: int, h: int, f: int) -> dict:
    return {
        'Wq': normal((n, d, h)),
        'bq': np.zeros((n, h)),
        'Wk': normal((n, d, h)),
        'bk': np.zeros((n, h)),
        'Wv': normal((n, d, h)),
        'bv': np.zeros((n, h)),
        'Wo': normal((n, h, d)),
        'bo': np.zeros(d),
        'ln1_g': np.ones(d),
        'ln1_b': np.zeros(d),
        'ln2_g': np.ones(d),
        'ln2_b': np.zeros(d),
        'W1': normal((d, f)),
        'b1': np.zeros(f),
        'W2': normal((f, d)),
        'b2': np.zeros(d),
    }

layer_weights = [init_block(D, N, H, F) for _ in range(L)]
tok_embed = normal((V, D))
pos_enc = normal((64, D))
final_ln_g = np.ones(D)
final_ln_b = np.zeros(D)

# explicit tied output projection
unembed = tok_embed.T

## 5. Algebra forward pass via `arch.interpreter('Transformer')`

`arch.interpreter('Transformer', params=...)` returns an `ArchInterpreter`.
Calling `.run_algebra()` uses the algebra functor with per-case cells.
Compiled paths are injected into `params['paths']` automatically.

In [6]:
def build_program(blocks, x0):
    """Build the algebra tree: input -> (attn + ffn) per layer -> final norm."""
    node = ('input', [x0], [])
    for w in blocks:
        node = ('ffn_residual', [w], [
            ('attn_residual', [w], [node])
        ])
    node = ('final_norm', [final_ln_g, final_ln_b], [node])
    return node

def transformer_forward(token_ids):
    t = len(token_ids)
    x0 = tok_embed[token_ids] + pos_enc[:t]
    interp = arch.interpreter('Transformer', params={'mask': causal_mask(t)})
    x = interp.run_algebra(build_program(layer_weights, x0), lambda n: n)
    return x @ unembed

token_ids = rng.integers(0, V, size=S)
logits = transformer_forward(token_ids)
print("token_ids:", token_ids)
print("logits shape:", logits.shape)

token_ids: [  0  27  72  83  50  41 104  94]
logits shape: (8, 128)


## 6. Path trace

In [7]:
x0 = tok_embed[token_ids] + pos_enc[:S]
w0 = layer_weights[0]
bundle0 = _build_attn_bundle(arch.paths, x0, w0, causal_mask(S))

print("Attention path trace:")
for name, eq, shape, _ in arch.trace("read", x0, bundle0):
    print(f"  {name:<12} eq={str(eq):<16} shape={shape}")

print("\nFFN path trace:")
for name, eq, shape, _ in arch.trace("mlp", x0, w0):
    print(f"  {name:<12} eq={str(eq):<16} shape={shape}")

Attention path trace:
  input        eq=None             shape=(8, 48)
  q_proj       eq=sd,ndh->snh      shape=(8, 4, 12)
  score        eq=snh,tnh->nst     shape=(4, 8, 8)
  normalize    eq=nst->nst         shape=(4, 8, 8)
  mix          eq=nst,tnh->snh     shape=(8, 4, 12)
  out_proj     eq=snh,nhd->sd      shape=(8, 48)

FFN path trace:
  input        eq=None             shape=(8, 48)
  up           eq=sd,df->sf        shape=(8, 144)
  act          eq=sf->sf           shape=(8, 144)
  down         eq=sf,fd->sd        shape=(8, 48)


## 7. Coalgebra streaming via `arch.interpreter('Transformer')`

Same `ArchInterpreter`, now calling `.run_coalgebra()`. The coalgebra cell
is bound at the functor level (`cell = ops.stream_cell`) since it dispatches
on the event mode internally.

In [8]:
prompt = token_ids[:6].tolist()
events = [('cache', tok) for tok in prompt[:-1]] + [('emit', prompt[-1])]

interp = arch.interpreter('Transformer', params={
    'tok_embed': tok_embed,
    'pos_enc': pos_enc,
    'layer_weights': layer_weights,
    'final_ln_g': final_ln_g,
    'final_ln_b': final_ln_b,
    'unembed': unembed,
})

stream_outputs, final_state = interp.run_coalgebra(
    state={
        'pos': 0,
        'caches': [{'K': np.zeros((0, N, H)), 'V': np.zeros((0, N, H))} for _ in range(L)],
    },
    token_iter=events,
)

full_last = transformer_forward(np.array(prompt))[-1]
stream_last = stream_outputs[-1]

print("prompt:", prompt)
print("emitted outputs:", len(stream_outputs))
print("final position:", final_state['pos'])
print("max abs diff vs batch forward:", np.max(np.abs(stream_last - full_last)))

prompt: [0, 27, 72, 83, 50, 41]
emitted outputs: 1
final position: 6
max abs diff vs batch forward: 1.1102230246251565e-16


## 8. Summary

This notebook defines a multi-head, multi-layer transformer through a single DSL block:

```
arch Transformer:
    algebra:
        case input:         recursive=0  data=1  cell=ops.input_cell
        case attn_residual: recursive=1  data=1  cell=ops.attn_cell
        case ffn_residual:  recursive=1  data=1  cell=ops.ffn_cell
        case final_norm:    recursive=1  data=2  cell=ops.norm_cell
    coalgebra:
        cell = ops.stream_cell
        case cache_only: recursive=1  data=1  output=0
        case emit:       recursive=1  data=1  output=1
```

One call to `compile(DSL_SOURCE, namespace)` produces an `ArchDef`.
One call to `arch.interpreter('Transformer', params=...)` returns an `ArchInterpreter`
with both `.run_algebra()` and `.run_coalgebra()`.

The algebra/coalgebra agreement holds: max abs diff between batch forward and
streaming last-token logits is at machine epsilon.